# 🔗 Structural Alignment & Superposition

---

## Learning Objectives

- Understand structural alignment vs sequence alignment
- Calculate RMSD between structures
- Use superposition algorithms
- Compare protein structures quantitatively

---

## Structural vs Sequence Alignment

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    SEQUENCE vs STRUCTURE ALIGNMENT                      │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  SEQUENCE ALIGNMENT:           STRUCTURAL ALIGNMENT:                    │
│                                                                         │
│  MVLSPADKTN...                 [3D coordinates]                         │
│  || ||| |                          ╭──╮                                 │
│  MV-SPAD-TN...                    ╱    ╲  superimpose                   │
│                                  ╱  ╭───╲─╮  backbone                   │
│  Based on: letters             ╱  ╱      ╲ ╲                            │
│  Method: Dynamic prog.        ╱  ╱        ╲ ╲                           │
│  Output: %identity           ╱──╯          ╰──╲                         │
│                                                                         │
│                              Based on: 3D coordinates                   │
│                              Method: Least-squares fit                  │
│                              Output: RMSD (Ångströms)                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

**Key insight**: Proteins with <20% sequence identity can have nearly identical structures!

---

## RMSD (Root Mean Square Deviation)

RMSD measures the average distance between aligned atoms:

```
                    ┌─────────────────────────────────┐
                    │                                 │
                    │         _______________         │
                    │        ╱  Σ(dᵢ)²              │
                    │ RMSD = ╲╱ ─────────             │
                    │             N                   │
                    │                                 │
                    │  dᵢ = distance between          │
                    │       equivalent atoms i        │
                    │  N = number of atoms            │
                    │                                 │
                    └─────────────────────────────────┘

    Interpretation:
    ┌────────────┬──────────────────────────────────────┐
    │ RMSD (Å)   │ Interpretation                       │
    ├────────────┼──────────────────────────────────────┤
    │ 0 - 1      │ Nearly identical structures          │
    │ 1 - 2      │ Very similar (same fold)            │
    │ 2 - 3      │ Similar fold, some variation        │
    │ 3 - 5      │ Same topology, different details    │
    │ > 5        │ Different structures                │
    └────────────┴──────────────────────────────────────┘
```

In [ ]:
import numpy as np

def calculate_rmsd(coords1, coords2):
    """
    Calculate RMSD between two sets of coordinates.
    
    Parameters:
        coords1, coords2: Nx3 numpy arrays of coordinates
        
    Returns:
        RMSD value in same units as input (usually Ångströms)
    """
    if len(coords1) != len(coords2):
        raise ValueError("Coordinate arrays must have same length")
    
    diff = coords1 - coords2
    squared_distances = np.sum(diff ** 2, axis=1)
    rmsd = np.sqrt(np.mean(squared_distances))
    
    return rmsd

# Example
coords1 = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [2.0, 0.0, 0.0],
    [3.0, 0.0, 0.0]
])

coords2 = np.array([
    [0.1, 0.1, 0.0],
    [1.1, -0.1, 0.0],
    [1.9, 0.1, 0.1],
    [3.1, 0.0, -0.1]
])

rmsd = calculate_rmsd(coords1, coords2)
print(f"RMSD: {rmsd:.3f} Å")

---

## Kabsch Algorithm for Optimal Superposition

The Kabsch algorithm finds the optimal rotation to minimize RMSD:

1. Center both structures at origin
2. Calculate covariance matrix H = P^T × Q
3. Compute SVD: H = U × S × V^T
4. Rotation matrix: R = V × U^T
5. Apply rotation and calculate RMSD

In [ ]:
def kabsch_superposition(mobile, reference):
    """
    Superimpose mobile structure onto reference using Kabsch algorithm.
    
    Parameters:
        mobile: Nx3 array - structure to move
        reference: Nx3 array - target structure
        
    Returns:
        rotated_mobile: Nx3 array - superimposed mobile coordinates
        rmsd: RMSD after superposition
        rotation_matrix: 3x3 rotation matrix
    """
    # Center both structures
    mobile_center = np.mean(mobile, axis=0)
    ref_center = np.mean(reference, axis=0)
    
    mobile_centered = mobile - mobile_center
    ref_centered = reference - ref_center
    
    # Compute covariance matrix
    H = mobile_centered.T @ ref_centered
    
    # Singular Value Decomposition
    U, S, Vt = np.linalg.svd(H)
    
    # Handle reflection case
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1, 1, d])
    
    # Optimal rotation matrix
    R = Vt.T @ D @ U.T
    
    # Apply rotation and translation
    rotated_mobile = (mobile_centered @ R) + ref_center
    
    # Calculate RMSD
    rmsd = calculate_rmsd(rotated_mobile, reference)
    
    return rotated_mobile, rmsd, R

# Example with slightly rotated structure
np.random.seed(42)
reference = np.random.randn(20, 3) * 10  # Random "protein"

# Create rotated version
angle = np.radians(30)
rotation = np.array([
    [np.cos(angle), -np.sin(angle), 0],
    [np.sin(angle), np.cos(angle), 0],
    [0, 0, 1]
])
mobile = (reference @ rotation) + np.array([5, 3, 2])  # Rotate and translate

# Superimpose
aligned, rmsd, R = kabsch_superposition(mobile, reference)

print(f"RMSD before alignment: {calculate_rmsd(mobile, reference):.3f} Å")
print(f"RMSD after alignment: {rmsd:.6f} Å")
print("\nRotation matrix:")
print(R.round(4))

---

## TM-score: Template Modeling Score

TM-score is length-normalized and more robust than RMSD:

```
                    ┌──────────────────────────────────────┐
                    │                                      │
                    │           1      Σ        1          │
                    │ TM-score = ─── × ─── ────────────    │
                    │           Lref   i   1 + (dᵢ/d₀)²   │
                    │                                      │
                    │  d₀ = 1.24 × ∛(Lref - 15) - 1.8     │
                    │                                      │
                    └──────────────────────────────────────┘

    Interpretation:
    ┌────────────┬──────────────────────────────────────┐
    │ TM-score   │ Interpretation                       │
    ├────────────┼──────────────────────────────────────┤
    │ > 0.5      │ Same fold                           │
    │ 0.3 - 0.5  │ Possibly same fold                  │
    │ < 0.3      │ Different folds                     │
    │ ≈ 1.0      │ Nearly identical                    │
    └────────────┴──────────────────────────────────────┘
```

In [ ]:
def calculate_tm_score(coords1, coords2, L_ref=None):
    """
    Calculate TM-score between aligned structures.
    
    Parameters:
        coords1, coords2: Aligned Nx3 coordinate arrays
        L_ref: Reference length (default: length of coords)
    """
    if L_ref is None:
        L_ref = len(coords1)
    
    # Distance scaling factor
    d0 = 1.24 * (L_ref - 15) ** (1/3) - 1.8
    if d0 < 0.5:
        d0 = 0.5
    
    # Calculate distances
    distances = np.sqrt(np.sum((coords1 - coords2) ** 2, axis=1))
    
    # TM-score sum
    tm_sum = np.sum(1 / (1 + (distances / d0) ** 2))
    
    # Normalize
    tm_score = tm_sum / L_ref
    
    return tm_score

# Example
tm = calculate_tm_score(aligned, reference)
print(f"TM-score: {tm:.4f}")

---

## Jmol Superposition Commands

```javascript
// Load two structures
load files "1abc.pdb" "2xyz.pdb"

// Align structure 2 onto structure 1
compare {2.1} {1.1} ATOMS {CA} ROTATE TRANSLATE

// Color differently to distinguish
select 1.1
color blue
select 2.1
color red

// Show backbone only
backbone only
```

---

## 🏋️ Exercises

### Exercise 1: Compare Homologs
Download two homologous proteins and calculate their RMSD.

### Exercise 2: RMSD by Region
Calculate RMSD separately for different domains or secondary structure elements.

### Exercise 3: Superposition Script
Write a script to batch-superimpose multiple structures onto a reference.

---

## 📚 Resources

- [TM-align Server](https://zhanggroup.org/TM-align/)
- [FATCAT Server](https://fatcat.godziklab.org/)
- [DALI Server](http://ekhidna2.biocenter.helsinki.fi/dali/)